# Cross-Domain Sentiment Analysis: Mitigating Domain Shift

## 1. Project Overview
This project tackles **Domain Shift** in Sentiment Analysis. Sentiment models trained heavily on one type of text (e.g., formal reviews or short tweets) often fail when applied to a new domain (e.g., private chats or student emails). 

Our goal is to build a robust ensemble model that can generalize across diverse domains, specifically focusing on the shift from general social media (Twitter) to student-life contexts (Gmail, WhatsApp, and Google Play Store App Reviews).

### Key Objectives:
- **Training Domain:** General sentiment from social media (Twitter).
- **Target Domains (Domain Shift):** 
  1. Private WhatsApp messages (Student slang/chat).
  2. Student-related Gmail threads (Formal/Transactional).
  3. Google App Reviews (Product Feedback - Manually Labeled).
- **The Ensemble Approach:** Stacking 10 diverse models (RNNs + Transformers) to ensure high predictive stability across all domains.

## 2. Importing the necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import random
import gc
import zipfile
import urllib.request
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SpatialDropout1D, Conv1D, GlobalMaxPooling1D, LSTM, Bidirectional, GRU, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

import xgboost as xgb
from lime.lime_text import LimeTextExplainer
from matplotlib.patches import FancyBboxPatch

sns.set_theme(style="whitegrid", palette="muted")
gpus = tf.config.list_physical_devices('GPU')
if gpus: 
    try: tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=4096)])
    except RuntimeError as e: print(e)
SEED = 42
def seed_everything(seed=42):
    random.seed(seed) 
    os.environ['PYTHONHASHSEED'] = str(seed) 
    np.random.seed(seed)
    tf.random.set_seed(seed) 
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)
seed_everything(SEED); DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Data Loading

This section handles the loading of data at different stages of the pipeline: from raw source files for cleaning, to processed datasets for model training and evaluation.

In [ ]:
# Training data
df_sentiment = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/sentimentdataset.csv')
df_tweets = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/Tweets.csv')
df_twitterdataset = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/twitterdataset.csv', header=None)

df_twitterdataset.columns  = ["id",'topic',"sentiment","text"]


# Gmail and whatsapp data for testing the models
df_gmail = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/gmail_raw.csv")
df_whatsapp = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/whatsapp_raw.csv")


# Load clean Twitter training data
train_df = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/processed_training_dataset.csv').dropna()
val_df = pd.read_csv('/kaggle/input/datasets/manuellaotim/last-ride/processed_validation_datset.csv').dropna()

# Combine training and validation for the larger ensemble training pool
train_df = pd.concat([train_df, val_df]).reset_index(drop=True)

# Load ready-to-predict test datasets
df_gmail_test = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/gmail_test_data.csv")
df_whatsapp_test = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/whatsapp_test_data.csv")
df_app_reviews_test = pd.read_csv("/kaggle/input/datasets/manuellaotim/last-ride/labelled_app_reviews_test.csv")


print("Done with data loading...")

## 4. Exploratory Data Analysis (EDA)

## 4.1 EDA for Training Dataset (Raw)

In [ ]:
display(df_sentiment_raw.head())
print("Raw Twitter Sentiment Distribution:")
print(df_twitter_raw['sentiment'].value_counts())

## 4.2 EDA for the Test Data (Raw)

In [ ]:
display("--- Gmail (Raw) ---")
display(df_gmail_raw.head(3))
display("--- WhatsApp (Raw) ---")
display(df_whatsapp_raw.head(3))
display("--- App Reviews (Raw) ---")
display(df_app_reviews_raw.head(3))

In [ ]:
print("=== Gmail Info ===")
df_gmail_raw.info()
print("\n=== WhatsApp Info ===")
df_whatsapp_raw.info()
print("\n=== App Reviews Info ===")
df_app_reviews_raw.info()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sentiment_colors = {'positive': '#4CAF50', 'neutral': '#2196F3', 'negative': '#F44336'}
def get_colors(index):
    return [sentiment_colors.get(str(i).lower(), '#9E9E9E') for i in index]

df_gmail_raw['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=get_colors(df_gmail_raw['sentiment'].value_counts().index), edgecolor='black')
axes[0].set_title('Gmail Labels')

df_whatsapp_raw['sentiment'].value_counts().plot(kind='bar', ax=axes[1], color=get_colors(df_whatsapp_raw['sentiment'].value_counts().index), edgecolor='black')
axes[1].set_title('WhatsApp Labels')

if 'score' in df_app_reviews_raw.columns:
    df_app_reviews_raw['score'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='skyblue', edgecolor='black')
    axes[2].set_title('App Reviews Scores')

plt.suptitle('Class Distribution in Test Datasets (Pre-Processing)')
plt.show()

## 5. Data Cleaning And Preprocessing

## 5.2 Test Data Preprocessing and Cleaning

In [ ]:
df_gmail = df_gmail_raw.drop_duplicates(subset=['text']).reset_index(drop=True)
df_whatsapp = df_whatsapp_raw.drop_duplicates(subset=['text']).reset_index(drop=True)
if 'content' in df_app_reviews_raw.columns:
    df_app_reviews = df_app_reviews_raw.drop_duplicates(subset=['content']).reset_index(drop=True)
else:
    df_app_reviews = df_app_reviews_raw.copy()

print(f"Gmail cleaned: {len(df_gmail)}")
print(f"WhatsApp cleaned: {len(df_whatsapp)}")
print(f"App Reviews cleaned: {len(df_app_reviews)}")

In [ ]:
def clean_the_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'--- Forwarded message ---', '', text)
    text = re.sub(r'!!!|\?\?\?|@user', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_gmail['clean_text'] = df_gmail['text'].apply(clean_the_text)
df_whatsapp['clean_text'] = df_whatsapp['text'].apply(clean_the_text)
if 'content' in df_app_reviews.columns:
    df_app_reviews['clean_text'] = df_app_reviews['content'].apply(clean_the_text)
elif 'text' in df_app_reviews.columns:
    df_app_reviews['clean_text'] = df_app_reviews['text'].apply(clean_the_text)

# Note: df_app_reviews['sentiment'] will be provided via manual labeling in Excel

In [ ]:
display("--- Gmail Preview ---")
display(df_gmail[['text', 'clean_text']].head())
display("--- WhatsApp Preview ---")
display(df_whatsapp[['text', 'clean_text']].head())
if 'clean_text' in df_app_reviews.columns:
    display("--- App Reviews Preview ---")
    display(df_app_reviews[['clean_text']].head())

## 6. Feature Engineering

In [ ]:
def extract_meta_features(df):
    df = df.copy()
    df['exclamation_count'] = df['text'].apply(lambda x: str(x).count('!'))
    df['question_count'] = df['text'].apply(lambda x: str(x).count('?'))
    df['is_all_caps'] = df['text'].apply(lambda x: 1 if str(x).isupper() and len(str(x)) > 5 else 0)
    df['char_cnt'] = df['text'].apply(lambda x: len(str(x)))
    df['word_cnt'] = df['text'].apply(lambda x: len(str(x).split()))
    
    platforms = r'github|slack|coursera|udemy|paystack|railway|netlify|heroku|mtn|airtel|gmail|whatsapp'
    alerts = r'invoice|billing|service termination|payment receipt|account alert|reminder notice|transaction'
    academic = r'assignment|deadline|exam|results|semester|lecture|submission|grade|marks|course'
    
    df['has_platform_mention'] = df['text'].apply(lambda x: 1 if re.search(platforms, str(x).lower()) else 0)
    df['has_service_alert'] = df['text'].apply(lambda x: 1 if re.search(alerts, str(x).lower()) else 0)
    df['exclamation_intensity'] = df['text'].apply(lambda x: min(str(x).count('!'), 5))
    df['technical_success'] = df['text'].apply(lambda x: 1 if re.search(r'successful|approved|passed|accepted|delivered|congratulations|internship|scholarship|live|verified|welcome|registered|great|amazing|won|celebrate', str(x).lower()) else 0)
    df['technical_failure'] = df['text'].apply(lambda x: 1 if re.search(r'failed|rejected|declined|suspended|expired|terminated|error|down|awful|hate|tired|annoying|struggling|frustrated|devastated|embarrassing|breaking', str(x).lower()) else 0)
    df['student_context_score'] = df['text'].apply(lambda x: len(re.findall(academic, str(x).lower())))
    df['positive_signal'] = df['text'].apply(lambda x: len(re.findall(r'congrats|congratulations|proud|excited|happy|amazing|passed|accepted|scholarship|won|celebrate|excellent|well done', str(x).lower())))
    return df

def surgical_cleaner(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = text.translate(str.maketrans('', '', ',.;:()[]{}<>/@#'))
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "notification"

# Ensure test sets have correct column names for features
df_gmail = df_gmail.rename(columns={'clean_text': 'text'})
df_whatsapp = df_whatsapp.rename(columns={'clean_text': 'text'})
if 'clean_text' in df_app_reviews.columns:
    df_app_reviews = df_app_reviews.rename(columns={'clean_text': 'text'})

train_df = extract_meta_features(full_train_df) # Using full training pool
df_gmail = extract_meta_features(df_gmail)
df_whatsapp = extract_meta_features(df_whatsapp)
df_app_reviews = extract_meta_features(df_app_reviews)

train_df['clean'] = train_df['text'].apply(surgical_cleaner)
df_gmail['clean'] = df_gmail['text'].apply(surgical_cleaner)
df_whatsapp['clean'] = df_whatsapp['text'].apply(surgical_cleaner)
df_app_reviews['clean'] = df_app_reviews['text'].apply(surgical_cleaner)

metadata_columns = ['exclamation_count', 'question_count', 'is_all_caps', 'char_cnt', 'word_cnt', 'has_platform_mention', 'has_service_alert', 'exclamation_intensity', 'technical_success', 'technical_failure', 'student_context_score', 'positive_signal']

scaler = StandardScaler() 
X_train_meta = scaler.fit_transform(train_df[metadata_columns]) 
X_gmail_meta = scaler.transform(df_gmail[metadata_columns])
X_whatsapp_meta = scaler.transform(df_whatsapp[metadata_columns])
X_app_meta = scaler.transform(df_app_reviews[metadata_columns])

VOCAB_SIZE, MAX_LEN = 20000, 150
deeplearning_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
deeplearning_tokenizer.fit_on_texts(pd.concat([train_df['clean'], df_gmail['clean'], df_whatsapp['clean'], df_app_reviews['clean']]))

## 7. Ensemble Model Training & Evaluation

In [ ]:
class SentiDS(torch.utils.data.Dataset):
        def __init__(self, enc, lbl): self.enc = enc; self.lbl = lbl
        def __getitem__(self, idx): 
            item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
            item['labels'] = torch.tensor(self.lbl[idx]); return item
        def __len__(self): return len(self.lbl)

def train_distilbert(train_txt, train_lbl):
    from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    train_enc = tokenizer(train_txt.tolist(), truncation=True, padding=True, max_length=128)
    model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', 
        num_labels=3
        ).to(DEVICE)
    
    args = TrainingArguments(
        output_dir='results', 
        num_train_epochs=3, 
        per_device_train_batch_size=8, 
        gradient_accumulation_steps=4, 
        learning_rate=2e-5, 
        warmup_ratio=0.1, 
        weight_decay=0.01, 
        dataloader_pin_memory=False, 
        fp16=True, 
        disable_tqdm=True, 
        save_strategy='no',
        report_to='none',
        logging_strategy='no')
    
    trainer = Trainer(
        model=model, 
        args=args, 
        train_dataset=SentiDS(train_enc, train_lbl))
    trainer.train() 
    return model, tokenizer

## 8. Comparative Analysis & Performance Comparison

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc

# ------------------------------------------------------------------
# 0. Training Performance (Source Domain: Twitter)
# ------------------------------------------------------------------
# oof_acc = accuracy_score(y_train, np.argmax(train_predictions, axis=1))
# print(f"Training Accuracy: {oof_acc*100:.2f}%")

# 1. Calculate Individual Domain Accuracies
# [acc_gmail, acc_whatsapp, acc_app calculated here]

# 2. Domain Comparison Visuals
# [Table, Bar Chart, Confusion Matrices]

## 9. Model Interpretability (LIME Stories)

In [ ]:
# [LIME explanation logic here]